# 键盘采集示教数据

在给定环境中采集示教数据。
任务是抓起杯子并放到盘子上。当杯子在盘子上、夹爪打开且末端执行器位于杯子上方时，环境会判定成功。

<img src="./media/teleop.gif" width="480" height="360">

键位说明：WASD 控制 x-y 平面移动，R/F 控制 z 轴，Q/E 控制倾斜，方向键控制其余旋转。

空格键切换夹爪开合状态，Z 键会重置环境并丢弃当前回合缓存数据。

叠加图像说明：
- 右上：Agent 视角
- 右下：腕部（第一人称）视角
- 左上：侧视图
- 左下：俯视图


注意采集数据时，尽量避免像视频中那样操作，需更精准地采集。

In [ ]:
# %pip install -r requirements.txt
# %pip install -e C:/tbxr_real
# %pip install robot_descriptions


In [11]:
import env_config

In [12]:
import sys
import time
import math
import threading
import numpy as np
import os
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image

import mujoco
import mujoco.viewer

from lerobot.common.datasets.lerobot_dataset import LeRobotDataset


In [13]:
# If you want to randomize object positions, set SEED=None
SEED = 0
# SEED = None

REPO_NAME = 'datawhale_eai_franka_pnp_xr'
NUM_DEMO = 1
ROOT = './demo_data_franka_pnp_xr'

# XR server
TELEOP_HOST = '0.0.0.0'
TELEOP_PORT = 4444

# Runtime
SIM_MODE = 'physics'   # 'physics' for pnp data; 'kinematic' for max responsiveness
SIM_FPS = 60
DATASET_FPS = 20
CAPTURE_EVERY_N = max(1, SIM_FPS // DATASET_FPS)


In [14]:
# TeleopXR source path
TELEOPXR_ROOT = Path(r"C:\tbxr_real")
assert TELEOPXR_ROOT.exists(), 'C:\tbxr_real not found. Check TeleopXR source path.'
if str(TELEOPXR_ROOT) not in sys.path:
    sys.path.insert(0, str(TELEOPXR_ROOT))

from teleop_xr import Teleop
from teleop_xr.config import TeleopSettings
from teleop_xr.messages import XRState, XRDeviceRole, XRHandedness
from teleop_xr.ik.robots.franka import FrankaRobot
from teleop_xr.ik.solver import PyrokiSolver
from teleop_xr.ik.controller import IKController
import teleop_xr

from robot_descriptions import panda_mj_description

# relax pose-jump threshold to reduce unwanted reset
_orig_are_close = teleop_xr.are_close
def _are_close_relaxed(a, b=None, lin_tol=1e-9, ang_tol=1e-9):
    return _orig_are_close(a, b, lin_tol=0.20, ang_tol=math.radians(80))
teleop_xr.are_close = _are_close_relaxed


记得解压这个文件夹

![fig1](./media/fig1.png)


In [15]:
TASK_NAME = 'Put mug cup on the plate'

state_lock = threading.Lock()
shared = {'latest_state': None}


def on_xr_state(_pose, info_dict):
    try:
        state = XRState.model_validate(info_dict)
    except Exception:
        return
    with state_lock:
        shared['latest_state'] = state


teleop = Teleop(TeleopSettings(host=TELEOP_HOST, port=TELEOP_PORT, input_mode='controller'))
teleop.subscribe(on_xr_state)
server_thread = threading.Thread(target=teleop.run, daemon=True)
server_thread.start()

robot = FrankaRobot()
solver = PyrokiSolver(robot)
controller = IKController(robot, solver)

# deadman: right squeeze only

def _deadman_right_only(self, state):
    for d in state.devices:
        if d.role == XRDeviceRole.CONTROLLER and d.handedness == XRHandedness.RIGHT:
            if d.gamepad is None:
                return False
            return (len(d.gamepad.buttons) > 1 and bool(d.gamepad.buttons[1].pressed))
    return False


controller._check_deadman = _deadman_right_only.__get__(controller, type(controller))


def _clone_elem(elem):
    return ET.fromstring(ET.tostring(elem, encoding='utf-8'))


def _abs_file_attr(xml_dir: Path, elem: ET.Element):
    f = elem.get('file')
    if not f:
        return
    fp = Path(f)
    if not fp.is_absolute():
        elem.set('file', os.path.abspath(str(xml_dir / fp)).replace('\\', '/'))


def _merge_xml_sections(dst_root: ET.Element, src_root: ET.Element, src_xml_path: Path):
    # Fix file paths first
    for tag in ('mesh', 'texture', 'hfield'):
        for e in src_root.findall(f'.//{tag}'):
            _abs_file_attr(src_xml_path.parent, e)

    # Merge common sections
    for sec in ('asset', 'default', 'worldbody', 'actuator', 'tendon', 'equality', 'contact', 'sensor'):
        src_sec = src_root.find(sec)
        if src_sec is None:
            continue
        dst_sec = dst_root.find(sec)
        if dst_sec is None:
            dst_sec = ET.SubElement(dst_root, sec)
        for child in list(src_sec):
            dst_sec.append(_clone_elem(child))


def _build_franka_pnp_scene_xml():
    project_root = Path.cwd()
    asset_root = project_root / 'asset'

    # Avoid non-ASCII path issues on Windows: create ASCII junction C:/tbxr_runtime/asset -> <project>/asset
    safe_root = Path('C:/tbxr_runtime')
    safe_asset_root = safe_root / 'asset'
    asset_root_for_xml = asset_root
    if os.name == 'nt':
        import subprocess
        safe_root.mkdir(parents=True, exist_ok=True)
        if (not safe_asset_root.exists()) and asset_root.exists():
            cmd = f'mklink /J "{safe_asset_root}" "{asset_root}"'
            subprocess.run(['cmd', '/c', cmd], check=False, capture_output=True)
        if safe_asset_root.exists():
            asset_root_for_xml = safe_asset_root

    panda_xml = Path(panda_mj_description.MJCF_PATH)
    tree = ET.parse(panda_xml)
    root = tree.getroot()

    # Remove keyframes because added free joints change nq.
    keyframe = root.find('keyframe')
    if keyframe is not None:
        root.remove(keyframe)

    compiler = root.find('compiler')
    if compiler is None:
        compiler = ET.SubElement(root, 'compiler')
    compiler.set('angle', 'radian')
    compiler.set('meshdir', str((panda_xml.parent / 'assets').resolve()).replace('\\', '/'))
    compiler.set('autolimits', 'true')

    # Put Franka on table height (~0.8m).
    worldbody = root.find('worldbody')
    if worldbody is None:
        worldbody = ET.SubElement(root, 'worldbody')
    for b in worldbody.findall('body'):
        if b.get('name') in ('link0', 'panda_link0', 'base', 'panda'):
            b.set('pos', '0 0 0.82')

    # Merge original assets from tutorial scene.
    include_files = [
        asset_root_for_xml / 'tabletop' / 'object' / 'floor_isaac_style.xml',
        asset_root_for_xml / 'tabletop' / 'object' / 'object_table.xml',
        asset_root_for_xml / 'objaverse' / 'mug_5' / 'model_new.xml',
        asset_root_for_xml / 'objaverse' / 'plate_11' / 'model_new.xml',
    ]
    for inc in include_files:
        src_tree = ET.parse(inc)
        src_root = src_tree.getroot()
        _merge_xml_sections(root, src_root, inc)

    # Add wrist camera if missing.
    hand = None
    for b in root.findall('.//body'):
        if b.get('name') in ('hand', 'panda_hand', 'tcp_link'):
            hand = b
            if b.get('name') == 'hand':
                break
    if hand is not None and not any((c.get('name') == 'egocentric') for c in hand.findall('camera')):
        ET.SubElement(hand, 'camera', name='egocentric', pos='0.0 -0.06 0.05', xyaxes='-1 0 0 0 0 1', fovy='90')

    out_dir = safe_root / 'generated'
    out_dir.mkdir(parents=True, exist_ok=True)
    out_xml = out_dir / 'franka_pnp_scene.xml'
    tree.write(out_xml, encoding='utf-8', xml_declaration=False)
    return out_xml


scene_xml = _build_franka_pnp_scene_xml()


{"host":"0.0.0.0","port":4444,"robot_vis":null,"input_mode":"controller","speed":1.0,"natural_phone_orientation_euler":[0.0,-0.7853981633974483,0.0],"natural_phone_position":[0.0,0.0,0.0],"camera_views":{},"video_config":null}
Server started at 0.0.0.0:4444
The phone web app should be available at https://192.168.209.36:4444
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 4444): 通常每个套接字地址(协议/网络地址/端口)只允许使用一次。
Exception ignored in: <function _xla_gc_callback at 0x00000191CE812F80>
Traceback (most recent call last):
  File "C:\Users\kewei\micromamba\envs\py310\lib\site-packages\jax\_src\lib\__init__.py", line 119, in _xla_gc_callback
    xla_client._xla.collect_garbage()
KeyboardInterrupt: 
Exception ignored in: <function _xla_gc_callback at 0x00000191CE812F80>
Traceback (most recent call last):
  File "C:\Users\kewei\micromamba\envs\py310\lib\site-packages\jax\_src\lib\__init__.py", line 119, in _xla_gc_callback
    xla_client._xla.collect_garbage()
Keyboard

In [16]:
model = mujoco.MjModel.from_xml_path(str(scene_xml))
data = mujoco.MjData(model)

ValueError: Error: resource not found via provider or OS filesystem: 'C:/Users/kewei/Documents/2025/04资料整理/03具身教程编写/ai-hardware-robotics/06-策略抓取或抓取VLA/大模型控制、VLA、VLM/04mujoco复现ACT、Pi0、SmolVLA/asset/objaverse/mug_5/visual/model_normalized_0.obj'

In [ ]:



# joint mapping

def map_joint_qpos_indices(model, target_joint_names):
    model_joint_names = [mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, i) for i in range(model.njnt)]
    model_joint_set = set(model_joint_names)
    idx, valid_names = [], []
    for name in target_joint_names:
        candidates = [name]
        if name.startswith('panda_'):
            candidates.append(name.replace('panda_', '', 1))
        else:
            candidates.append('panda_' + name)
        if 'joint' in name:
            candidates.append(name[name.find('joint'):])
        hit = None
        for c in candidates:
            if c in model_joint_set:
                hit = c
                break
        if hit is not None:
            jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, hit)
            idx.append(int(model.jnt_qposadr[jid]))
            valid_names.append(hit)
    return valid_names, idx


ik_arm_joint_names = list(robot.actuated_joint_names[:7])
arm_joint_names, arm_qpos_idx = map_joint_qpos_indices(model, ik_arm_joint_names)
gripper_candidates = ['panda_finger_joint1', 'panda_finger_joint2', 'finger_joint1', 'finger_joint2']
gripper_names, gripper_qpos_idx = map_joint_qpos_indices(model, gripper_candidates)
assert len(arm_qpos_idx) == 7, f'arm mapping failed: {len(arm_qpos_idx)}'


# object joints
mug_free_jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'mug_free')
plate_free_jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'plate_free')
mug_qadr = int(model.jnt_qposadr[mug_free_jid]) if mug_free_jid >= 0 else -1
plate_qadr = int(model.jnt_qposadr[plate_free_jid]) if plate_free_jid >= 0 else -1


def randomize_objects(seed=None):
    rng = np.random.default_rng(seed)
    for _ in range(200):
        mug_xy = np.array([rng.uniform(0.35, 0.55), rng.uniform(-0.20, 0.20)], dtype=np.float32)
        plate_xy = np.array([rng.uniform(0.35, 0.60), rng.uniform(-0.20, 0.20)], dtype=np.float32)
        if np.linalg.norm(mug_xy - plate_xy) >= 0.18:
            break
    if mug_qadr >= 0:
        data.qpos[mug_qadr:mug_qadr+3] = np.array([mug_xy[0], mug_xy[1], 0.86], dtype=np.float32)
        data.qpos[mug_qadr+3:mug_qadr+7] = np.array([1, 0, 0, 0], dtype=np.float32)
    if plate_qadr >= 0:
        data.qpos[plate_qadr:plate_qadr+3] = np.array([plate_xy[0], plate_xy[1], 0.82], dtype=np.float32)
        data.qpos[plate_qadr+3:plate_qadr+7] = np.array([1, 0, 0, 0], dtype=np.float32)
    mujoco.mj_forward(model, data)
    return np.concatenate([
        np.array([mug_xy[0], mug_xy[1], 0.86], dtype=np.float32),
        np.array([plate_xy[0], plate_xy[1], 0.82], dtype=np.float32),
    ], dtype=np.float32)


def get_trigger_value(xr_state):
    if xr_state is None:
        return 0.0
    for d in xr_state.devices:
        if d.role == XRDeviceRole.CONTROLLER and d.handedness == XRHandedness.RIGHT and d.gamepad is not None:
            if len(d.gamepad.buttons) > 0:
                return float(d.gamepad.buttons[0].value)
    return 0.0


def get_squeeze_pressed(xr_state, hand: str):
    if xr_state is None:
        return False
    for d in xr_state.devices:
        if d.role != XRDeviceRole.CONTROLLER:
            continue
        if hand == 'right' and d.handedness != XRHandedness.RIGHT:
            continue
        if hand == 'left' and d.handedness != XRHandedness.LEFT:
            continue
        if d.gamepad is None or len(d.gamepad.buttons) <= 1:
            return False
        return bool(d.gamepad.buttons[1].pressed)
    return False


# both hands double-squeeze to manually end an episode
reset_state = {
    'prev_left': False,
    'prev_right': False,
    'left_times': [],
    'right_times': [],
    'double_window_s': 0.40,
    'sync_window_s': 0.80,
}


def update_bimanual_double_squeeze(xr_state, now_s):
    l = get_squeeze_pressed(xr_state, 'left')
    r = get_squeeze_pressed(xr_state, 'right')
    if l and (not reset_state['prev_left']):
        reset_state['left_times'].append(now_s)
    if r and (not reset_state['prev_right']):
        reset_state['right_times'].append(now_s)
    reset_state['prev_left'] = l
    reset_state['prev_right'] = r

    dw = reset_state['double_window_s']
    reset_state['left_times'] = [t for t in reset_state['left_times'] if now_s - t <= 2.0]
    reset_state['right_times'] = [t for t in reset_state['right_times'] if now_s - t <= 2.0]

    left_double = len(reset_state['left_times']) >= 2 and (reset_state['left_times'][-1] - reset_state['left_times'][-2] <= dw)
    right_double = len(reset_state['right_times']) >= 2 and (reset_state['right_times'][-1] - reset_state['right_times'][-2] <= dw)

    if left_double and right_double:
        if abs(reset_state['left_times'][-1] - reset_state['right_times'][-1]) <= reset_state['sync_window_s']:
            reset_state['left_times'].clear()
            reset_state['right_times'].clear()
            return True
    return False


def apply_targets_to_mujoco(q_arm, gripper_open_ratio):
    n = min(len(q_arm), len(arm_qpos_idx))
    for i in range(n):
        data.qpos[arm_qpos_idx[i]] = q_arm[i]
    for jname, qidx in zip(gripper_names, gripper_qpos_idx):
        jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, jname)
        if jid >= 0:
            qmin, qmax = model.jnt_range[jid]
            data.qpos[qidx] = qmin + float(gripper_open_ratio) * (qmax - qmin)


def get_body_pos(body_name):
    bid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, body_name)
    if bid < 0:
        return None
    return data.xpos[bid].copy()


def check_success_like_collect():
    p_mug = get_body_pos('body_obj_mug_5')
    p_plate = get_body_pos('body_obj_plate_11')
    p_hand = get_body_pos('hand')
    if p_mug is None or p_plate is None or p_hand is None:
        return False

    near_xy = np.linalg.norm(p_mug[:2] - p_plate[:2]) < 0.10
    near_z = abs(p_mug[2] - p_plate[2]) < 0.10
    gripper_open = float(data.qpos[gripper_qpos_idx[0]] + data.qpos[gripper_qpos_idx[1]]) > 0.03 if len(gripper_qpos_idx) >= 2 else True
    ee_up = p_hand[2] > 0.75
    return bool(near_xy and near_z and gripper_open and ee_up)


def mat_to_rpy(R):
    sy = np.sqrt(R[0, 0] * R[0, 0] + R[1, 0] * R[1, 0])
    singular = sy < 1e-6
    if not singular:
        roll = np.arctan2(R[2, 1], R[2, 2])
        pitch = np.arctan2(-R[2, 0], sy)
        yaw = np.arctan2(R[1, 0], R[0, 0])
    else:
        roll = np.arctan2(-R[1, 2], R[1, 1])
        pitch = np.arctan2(-R[2, 0], sy)
        yaw = 0.0
    return np.array([roll, pitch, yaw], dtype=np.float32)


def render_cam(renderer, cam_name):
    renderer.update_scene(data, camera=cam_name)
    return renderer.render()


q_target = np.array(robot.get_default_config(), dtype=np.float32)
obj_init_pose = randomize_objects(SEED)

print('TeleopXR URL:', f'https://<LAN_IP>:{TELEOP_PORT}')
print('Scene XML:', scene_xml)
print('Using original assets: table + mug_5 + plate_11 + floor')
print('Cameras from scene: agentview + sideview + topview; wrist: egocentric')


## 定义数据集特征并创建你的数据集
数据集结构如下：
```
fps = 20,
features={
    "observation.image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.wrist_image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channel"],
    },
    "observation.state": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["state"], # x, y, z, roll, pitch, yaw
    },
    "action": {
        "dtype": "float32",
        "shape": (7,),
        "names": ["action"], # 6 joint angles and 1 gripper
    },
    "obj_init": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["obj_init"], # just the initial position of the object. Not used in training.
    },
},
```

数据将保存在 `./demo_data` 目录中。


In [ ]:
create_new = True
if os.path.exists(ROOT):
    print(f"Directory {ROOT} already exists.")
    ans = input("Do you want to delete it? (y/n) ")
    if ans == 'y':
        import shutil
        shutil.rmtree(ROOT)
    else:
        create_new = False

if create_new:
    dataset = LeRobotDataset.create(
                repo_id=REPO_NAME,
                root=ROOT,
                robot_type='franka',
                fps=DATASET_FPS,
                features={
                    "observation.image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channels"],
                    },
                    "observation.wrist_image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channels"],
                    },
                    "observation.state": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["state"],
                    },
                    "action": {
                        "dtype": "float32",
                        "shape": (8,),
                        "names": ["action"],
                    },
                    "obj_init": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["obj_init"],
                    },
                },
                image_writer_threads=10,
                image_writer_processes=5,
            )
else:
    print("Load from previous dataset")
    dataset = LeRobotDataset(REPO_NAME, root=ROOT)


## 键盘控制
你可以用键盘遥操作机械臂并采集数据
```
---------     -----------------------
   w       ->        backward
s  a  d        left   forward   right
---------      -----------------------
In x, y plane

---------
R: Moving Up
F: Moving Down
---------
In z axis

---------
Q: Tilt left
E: Tilt right
UP: Look Upward
Down: Look Donward
Right: Turn right
Left: Turn left
---------
For rotation

---------
SPACEBAR: Toggle Gripper
--------

---------
z: reset
--------
```
重置环境会删除当前示教回合的缓存数据，并重新开始采集。


关于 Rotation（旋转）部分的详细解释：
在机器人控制中，这部分通常对应 3D 空间中的 欧拉角（Euler Angles）：

Tilt (Q/E): 对应 Roll（翻滚角）。想象机械臂末端像拧螺丝一样左右倾斜。

Look Upward/Downward (UP/Down): 对应 Pitch（俯仰角）。控制机械臂末端的“头”抬起或低下。

Turn Left/Right (Left/Right): 对应 Yaw（偏航角）。控制机械臂末端在水平方向上左右摆动。

### 现在开始遥操作并采集数据

**要收到成功信号，你需要松开夹爪并将末端执行器上移到杯子上方。**


杯子和盘子中心在 XY 上要足够近（< 0.1）

夹爪要“真正打开”（rh_r1 < 0.1，这个很容易卡住）

末端执行器（tcp_link）要抬高到 z > 0.9

以上都满足后，才会触发 done=True

In [ ]:
renderer = mujoco.Renderer(model, 256, 256)
action = np.zeros(8, dtype=np.float32)
episode_id = 0
record_flag = False
step_count = 0

with mujoco.viewer.launch_passive(model, data) as viewer:
    while viewer.is_running() and episode_id < NUM_DEMO:
        tic = time.time()
        with state_lock:
            xr_state = shared['latest_state']

        # IK step
        if xr_state is not None:
            q_target = np.array(controller.step(xr_state, q_target), dtype=np.float32)

        deadman_active = bool(controller.active)
        trig = get_trigger_value(xr_state)
        gripper_open_ratio = float(np.clip(1.0 - trig, 0.0, 1.0))

        apply_targets_to_mujoco(q_target[:7], gripper_open_ratio)

        if SIM_MODE == 'kinematic':
            data.qvel[:] = 0.0
            if hasattr(data, 'qacc'):
                data.qacc[:] = 0.0
            mujoco.mj_forward(model, data)
        else:
            mujoco.mj_step(model, data)

        done = check_success_like_collect()
        manual_end = update_bimanual_double_squeeze(xr_state, time.time())

        # render three views
        agent_image = render_cam(renderer, 'agentview')
        wrist_image = render_cam(renderer, 'egocentric')
        side_image = render_cam(renderer, 'sideview')

        # state: [x,y,z,roll,pitch,yaw]
        bid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'hand')
        if bid < 0:
            bid = model.nbody - 1
        p = data.xpos[bid].copy()
        R = data.xmat[bid].reshape(3, 3).copy()
        ee_state = np.concatenate([p, mat_to_rpy(R)], dtype=np.float32)

        action = np.concatenate([q_target[:7], np.array([1.0 - gripper_open_ratio], dtype=np.float32)], dtype=np.float32)

        if deadman_active and (not record_flag):
            record_flag = True
            print(f"Start recording episode {episode_id + 1}")

        if record_flag and (step_count % CAPTURE_EVERY_N == 0):
            dataset.add_frame(
                {
                    "observation.image": np.asarray(Image.fromarray(agent_image).resize((256, 256))),
                    "observation.wrist_image": np.asarray(Image.fromarray(wrist_image).resize((256, 256))),
                    "observation.state": ee_state,
                    "action": action,
                    "obj_init": obj_init_pose.astype(np.float32),
                },
                task=TASK_NAME,
            )

        # end condition: success or bimanual-double-squeeze
        if record_flag and (done or manual_end):
            dataset.save_episode()
            episode_id += 1
            print(f"Episode saved. episode_id={episode_id}, reason={'success' if done else 'manual_end'}")

            record_flag = False
            controller.reset()
            q_target = np.array(robot.get_default_config(), dtype=np.float32)
            apply_targets_to_mujoco(q_target[:7], 1.0)
            obj_init_pose = randomize_objects(None if SEED is None else (SEED + episode_id))
            mujoco.mj_forward(model, data)

        viewer.sync()

        step_count += 1
        if step_count % SIM_FPS == 0:
            l_sq = get_squeeze_pressed(xr_state, 'left')
            r_sq = get_squeeze_pressed(xr_state, 'right')
            print(
                f"steps={step_count}, deadman={deadman_active}, trigger={trig:.2f}, "
                f"gripper={1.0-gripper_open_ratio:.2f}, l_sq={l_sq}, r_sq={r_sq}, "
                f"record={record_flag}, done={done}, manual_end={manual_end}, episodes={episode_id}"
            )

        elapsed = time.time() - tic
        time.sleep(max(0.0, 1.0 / SIM_FPS - elapsed))


这里相当于只用采集一条数据

In [ ]:
try:
    teleop.app
except Exception:
    pass
print('Close viewer done. Teleop server thread is daemon and will exit with kernel restart.')


In [8]:
# Clean up the images folder (optional)
# import shutil
# shutil.rmtree(dataset.root / 'images')


In [ ]:
print('Dataset root:', ROOT)
print('Repo name:', REPO_NAME)
print('Cameras rendered in loop: agentview, egocentric, sideview')
print('Dataset images used: observation.image(agentview), observation.wrist_image(egocentric)')
